In [ ]:
try:
    from openmdao.utils.notebook_utils import notebook_mode  # noqa: F401
except ImportError:
    !python -m pip install openmdao[notebooks]

# The `set_input_defaults` function


The `set_input_defaults` function in OpenMDAO is used to specify metadata for inputs that are promoted to the same name within a Group. This is necessary when multiple inputs within a Group are promoted to the same name, but their units or initial values differ. If `set_input_defaults` is not used in this scenario, OpenMDAO will raise an error during setup.

The function does not set the actual values of the inputs, only the metadata that will be used to create the [Auto IVC](../../../other_useful_docs/auto_ivc_api_translation.ipynb)
output connected to them. `set_input_defaults` is typically called during the model setup phase, within the setup method of a Group.


```{eval-rst}
    .. automethod:: openmdao.core.group.Group.set_input_defaults
        :noindex:
```

## Example

In this example, we have two components that promote the variable `x` but use different units.  Note that this will result in an error unless you resolve the ambiguity by calling `set_input_defaults`:


In [ ]:
import openmdao.api as om

p = om.Problem()
model = p.model

model.add_subsystem('C1', om.ExecComp('y = 3.*x', x={'val': 3., 'units': 'm'}), promotes=['x'])
model.add_subsystem('C2', om.ExecComp('y = 4.*x', x={'val': 4., 'units': 'cm'}), promotes=['x'])

try:
    p.setup()       
except Exception as err:
    print(str(err))

In [ ]:
try:
    p.setup()       
except Exception as err:
    assert(str(err) == "\nCollected errors for problem 'problem':\n   <model> <class Group>: The following inputs, ['C1.x', 'C2.x'], promoted to 'x', are connected but their metadata entries ['units', 'val'] differ. Call <group>.set_input_defaults('x', units=?, val=?), where <group> is the model to remove the ambiguity.")
else:
    raise RuntimeError("Exception expected.")    

In [ ]:
# units and value to use for 'x' are ambiguous.  This fixes that.
model.set_input_defaults('x', val=1., units='m')

p.setup()

p.run_model()

model.list_inputs(units=True)
model.list_outputs(units=True);


## How the `set_input_defaults` function differs from the `set_val` function

While both `set_input_defaults` and `set_val` deal with variable management in OpenMDAO, they have distinct purposes and are used in different contexts.

- `set_input_defaults` is used at the group level to define default metadata (units and initial value) for promoted inputs, specifically to resolve ambiguity when multiple inputs are promoted to the same name. This is crucial for inputs connected to the automatically generated `_auto_ivc` component.

  - Used to resolve inconsistencies between auto-IVC values.

  - Specifically used at the group level to specify metadata to be assumed when multiple inputs are promoted to the same name. This is required when the promoted inputs have differing units or values.

  - For example, if two inputs, C1.x and C2.x, are both promoted to x but have different units (e.g., 'ft' and 'inch'), calling `set_input_defaults('x', units='m', val=1.0)` on the parent group will resolve the ambiguity.


- `set_val` is used at the problem level to set the actual value of a variable, including inputs, outputs, and independent variables. It can handle unit conversions and set values for specific indices in array variables.

  - Used at the run script level to set the value of an input variable.

  - Can be used to set the value of a variable in a different unit than its declared unit, and OpenMDAO will perform the conversion.

  - Can be used to set specific indices or index ranges of array variables.

In essence, `set_input_defaults` helps OpenMDAO correctly determine the units and initial values of connected inputs during the setup phase, while `set_val` is used to directly manipulate variable values before or during a run.

*Key Differences*

-  *Scope*: 
   `set_input_defaults` is used at the group level to define default metadata for promoted inputs, while `set_val` is used at the problem level to set specific values for variables.

- *Purpose*: 
  `set_input_defaults` resolves ambiguities when multiple inputs are promoted to the same name, while `set_val` is used to assign values to variables.

- *Timing*: 
  `set_input_defaults` is typically called during the model setup phase, while `set_val` can be called before or during a run of the model.

Note that `set_input_defaults` does not currently work for discretes.  In this example, the intent is to set the material for all objects by promoting it as a discrete variable from each component and setting it via `set_input_defaults`.

In [ ]:
import math
import openmdao.api as om

density = {
    'steel': 7.85,  # g/cm^3
    'aluminum': 2.7  # g/cm^3
}

class SquarePlate(om.ExplicitComponent):
    
    def setup(self):
        self.add_discrete_input('material', 'steel')
       
        self.add_input('length', 1.0, units='cm')
        self.add_input('width', 1.0, units='cm')
        self.add_input('thickness', 1.0, units='cm')
        
        self.add_output('weight', 1.0, units='g')

    def compute(self, inputs, outputs, discrete_inputs, discrete_outputs):
        length = inputs['length']
        width = inputs['width']
        thickness = inputs['thickness']
        material = discrete_inputs['material']
        
        print(f"square plate {material=}")
        outputs['weight'] = length * width * thickness * density[material]

class CirclePlate(om.ExplicitComponent):
    
    def setup(self):
        self.add_discrete_input('material', 'aluminum')
       
        self.add_input('radius', 1.0, units='cm')
        self.add_input('thickness', 1.0, units='g')
        
        self.add_output('weight', 1.0, units='g')

    def compute(self, inputs, outputs, discrete_inputs, discrete_output):
        radius = inputs['radius']
        thickness = inputs['thickness']
        material = discrete_inputs['material']                

        print(f"round plate {material=}")
        outputs['weight'] =  math.pi * radius**2 * thickness * density[material]

            
p = om.Problem()
model = p.model

model.add_subsystem('square', SquarePlate(), promotes_inputs=['material'])
model.add_subsystem('circle', CirclePlate(), promotes_inputs=['material'])

model.set_input_defaults('material', 'steel')
p.setup()

p.run_model()

model.list_inputs(units=True)
model.list_outputs(units=True);
            

Another possible scenario is to have multiple inputs promoted to the same name when those inputs have
different units, but then connecting them manually to an output using the `connect` function.
In this case, the framework will not raise an exception during setup if `set_input_defaults` was not
called as it does in the case of multiple promoted inputs that connected to `_auto_ivc`.  However,
if the user attempts to set or get the input using the promoted name, the framework *will* raise an
exception if `set_input_defaults` has not been called to disambiguate the units of the promoted
input.  The reason for this difference is that in the unconnected case, the framework won't know
what value and units to assign to the `_auto_ivc` output if they're ambiguous.  In the manually
connected case, the value and units of the output have already been supplied by the user, and
the only time that there's an ambiguity is if the user tries to access the inputs using their
promoted name.

Specifying Units

You can also set an input or request the value of any variable in a different unit than its declared
unit, and OpenMDAO will
perform the conversion for you. This is done with the `Problem` methods `get_val` and `set_val`.